# 06 - Évaluation et métriques

Objectif : calculer les métriques du livrable.

Fichiers concernés : `data/synthetic_cases.csv`, `src/inference.py`, `src/guardrails.py`, `src/metrics.py`, `outputs/`.

## Lire les métriques correctement

- Accuracy: proportion de classes prédites égales aux labels attendus.
- JSON valid rate: part des sorties respectant le contrat JSON.
- Warning rate: part des sorties contenant le warning obligatoire.
- Uncertain rate: part des sorties prudentes en `uncertain`.
- Latence médiane: temps typique de prédiction dans ce pipeline jouet.
- Precision: parmi les prédictions d'une classe, proportion correcte.
- Recall / sensibilité: parmi les vrais cas d'une classe, proportion retrouvée.
- F1: équilibre precision/recall.
- Spécificité: capacité à ne pas prédire une classe quand elle est absente.
- Matrice de confusion: répartition attendue/prédite.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX


In [2]:
import pandas as pd, time
from pathlib import Path
from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails, validate_prediction

classes = ["normal", "suspected_opacity", "uncertain"]
df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
rows = []
for _, row in df.iterrows():
    start = time.perf_counter()
    pred = apply_safety_guardrails(toy_predict(PROJECT_ROOT / row["image_path"], mode="baseline"))
    latency_ms = round((time.perf_counter() - start) * 1000, 3)
    valid, errors = validate_prediction(pred)
    rows.append({"case_id": row["case_id"], "filename": Path(row["image_path"]).name, "expected_label": row["label"], "predicted_class": pred["predicted_class"], "confidence": pred["confidence"], "json_valid": valid, "warning_present": bool(pred.get("warning")), "latency_ms": pred.get("latency_ms", latency_ms), "guardrail_errors": ";".join(errors)})
pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUTPUTS_DIR / "evaluation_predictions.csv", index=False)
display(pred_df.head())

,case_id,filename,expected_label,predicted_class,confidence,json_valid,warning_present,latency_ms,guardrail_errors
0,CXR_SYN_001,CXR_SYN_001_normal.png,normal,normal,0.72,True,True,0,
1,CXR_SYN_002,CXR_SYN_002_suspected_opacity.png,suspected_opacity,suspected_opacity,0.78,True,True,0,
2,CXR_SYN_003,CXR_SYN_003_uncertain.png,uncertain,uncertain,0.52,True,True,0,
3,CXR_SYN_004,CXR_SYN_004_normal.png,normal,normal,0.72,True,True,0,
4,CXR_SYN_005,CXR_SYN_005_suspected_opacity.png,suspected_opacity,suspected_opacity,0.78,True,True,0,


## Interprétation prudente

Un score parfait sur ces 30 cas valide surtout le pipeline technique: lecture CSV,
inférence jouet, garde-fous, exports et calculs de métriques. Il ne prouve pas une
capacité de lecture radiologique. Le résultat est attendu car les données sont
synthétiques et la baseline est déterministe avec label leakage par nom de fichier.

Dans le rapport, séparer explicitement:
- validation technique du logiciel;
- performance modèle réelle, non démontrée ici;
- validation clinique, hors périmètre.

In [3]:
def safe_div(a, b): return a / b if b else 0.0
y_true, y_pred = pred_df["expected_label"], pred_df["predicted_class"]
metrics_summary = pd.DataFrame([{"n": len(pred_df), "accuracy": round((y_true == y_pred).mean(), 4), "json_valid_rate": round(pred_df["json_valid"].mean(), 4), "warning_rate": round(pred_df["warning_present"].mean(), 4), "uncertain_rate": round((y_pred == "uncertain").mean(), 4), "latency_median_ms": round(pred_df["latency_ms"].median(), 4)}])
metrics_summary.to_csv(OUTPUTS_DIR / "metrics_summary.csv", index=False)
confusion = pd.crosstab(y_true, y_pred, rownames=["expected_label"], colnames=["predicted_class"]).reindex(index=classes, columns=classes, fill_value=0)
confusion.to_csv(OUTPUTS_DIR / "confusion_matrix.csv")
per_class, spec = [], []
for klass in classes:
    tp = int(((y_true == klass) & (y_pred == klass)).sum()); fp = int(((y_true != klass) & (y_pred == klass)).sum())
    fn = int(((y_true == klass) & (y_pred != klass)).sum()); tn = int(((y_true != klass) & (y_pred != klass)).sum())
    precision = safe_div(tp, tp + fp); recall = safe_div(tp, tp + fn); f1 = safe_div(2 * precision * recall, precision + recall)
    per_class.append({"class": klass, "tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 4), "recall_sensitivity": round(recall, 4), "f1": round(f1, 4)})
    spec.append({"class": klass, "tn": tn, "fp": fp, "specificity": round(safe_div(tn, tn + fp), 4)})
pd.DataFrame(per_class).to_csv(OUTPUTS_DIR / "per_class_metrics.csv", index=False)
pd.DataFrame(spec).to_csv(OUTPUTS_DIR / "specificity_metrics.csv", index=False)
display(metrics_summary); display(confusion); display(pd.DataFrame(per_class)); display(pd.DataFrame(spec))

,n,accuracy,json_valid_rate,warning_rate,uncertain_rate,latency_median_ms
0,30,1.0,1.0,1.0,0.3333,0.0


predicted_class,normal,suspected_opacity,uncertain
expected_label,,,
normal,10,0,0
suspected_opacity,0,10,0
uncertain,0,0,10


,class,tp,fp,fn,precision,recall_sensitivity,f1
0,normal,10,0,0,1.0,1.0,1.0
1,suspected_opacity,10,0,0,1.0,1.0,1.0
2,uncertain,10,0,0,1.0,1.0,1.0


,class,tn,fp,specificity
0,normal,20,0,1.0
1,suspected_opacity,20,0,1.0
2,uncertain,20,0,1.0


Conclusion : les métriques sont utiles pour le livrable si elles sont présentées
comme des métriques de pipeline. Elles devront être transférées ou consolidées dans
`src/metrics.py`, avec une explication claire du label leakage et des limites
synthétiques.